<a href="https://colab.research.google.com/github/Shashanth571/python-projects-/blob/main/PDF_summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pypdf

In [ ]:
import os
import json
from google import genai
from pypdf import PdfReader
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
system_prompt =  """You are a document summarizer given a document , extract :
-"summary": a single sentence summarizing the whole document
-"key_points": a list of 3-4 key points
-"action_items": a list of action items mentioned (empty list if none)

Responde ONLy with the valid JSON matching the exact structure. No markdown , no code fences, no explanation."""

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sample_meeting_notes.pdf to sample_meeting_notes (1).pdf


In [ ]:
filename = "sample_meeting_notes.pdf"
_, extension = os.path.splitext(filename)

if extension == ".pdf":
    reader = PdfReader(filename)
    file_content = ""
    for page in reader.pages:
        file_content += page.extract_text()
else:
    with open(filename, "r", encoding="utf-8") as file:
        file_content = file.read()

print(file_content[:1000])

Project Kickoff Meeting Notes
The team met to discuss the launch plan for the new mobile onboarding flow. Marketing confirmed the
campaign budget of $50,000 has been approved by finance. The design team presented final mockups,
which received sign-off from all stakeholders. Engineering flagged that the QA team will need
approximately 3 additional days to complete regression testing before the release can be considered
stable. There was also discussion about server capacity, and the infrastructure team confirmed current
capacity is sufficient for the expected launch traffic.
Next Steps
Sarah will assign a dedicated QA lead for the regression testing cycle by Friday. Marketing will send the
finalized launch email draft to the leadership team for review. Engineering will schedule a load-testing
session before the final go-live date. The whole team will reconvene next Tuesday to confirm final
launch readiness.



In [ ]:
if not os.environ.get("GEMINI_API_KEY"):
    raise ValueError("Oops! GEMINI_API_KEY is still missing from the environment. Check your .env file spelling.")

client = genai.Client()

In [ ]:
try:
    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        input =system_prompt + "\n\nDocument:\n" + file_content
    )

    raw = interaction.output_text.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").replace("json","",1).strip()

    data = json.loads(raw)

    print(f"Summary: {data['summary']}\n")
    print("Key Points:")
    for point in data["key_points"]:
        print(f"  - {point}")

    print("\nAction Items:")
    if data["action_items"]:
        for item in data["action_items"]:
            print(f"  - {item}")
    else:
        print("  (none)")

except json.JSONDecodeError as e:
    print(f"Failed to parse JSON : {e}")
    print("Raw response was: ")
    print(raw)

Summary: The team aligned on the launch plan for the new mobile onboarding flow, confirming budget approval, design sign-off, and server capacity readiness while outlining key pre-launch testing steps.

Key Points:
  - Finance approved a $50,000 campaign budget for the marketing launch.
  - Stakeholders officially signed off on the final design mockups presented by the design team.
  - Engineering noted that the QA team will require an additional three days for regression testing, but server capacity is confirmed sufficient.

Action Items:
  - Sarah to assign a dedicated QA lead for the regression testing cycle by Friday.
  - Marketing to send the finalized launch email draft to the leadership team for review.
  - Engineering to schedule a load-testing session before the final go-live date.
  - The entire team to reconvene next Tuesday to confirm final launch readiness.
